# sbarchi_migranti_italia - notebook v0

Validazione rapida + analisi esplorativa della pipeline **raw → clean → mart**.

- i check tecnici (righe, null, readiness) sono delegati al toolkit
- le SQL sono la fonte di verita: leggile prima di interpretare i numeri
- il notebook serve l'analisi, non sostituisce la validazione della pipeline
- evitare output pesanti o immagini embeddate nel commit

In [ ]:
import json, subprocess, sys, duckdb, yaml
from pathlib import Path

SLUG = "sbarchi_migranti_italia"

_start = Path.cwd().resolve()
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        CANDIDATE_DIR = probe
        break
else:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

CFG_PATH = CANDIDATE_DIR / "dataset.yml"
SQL_DIR = CANDIDATE_DIR / "sql"
_cfg_raw = yaml.safe_load(CFG_PATH.read_text())
ANNO = _cfg_raw["dataset"]["years"][-1]

def tk(*args: str) -> dict:
    cmd = ["toolkit", *args, "-c", str(CFG_PATH), "--json"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr, file=sys.stderr)
        r.check_returncode()
    return json.loads(r.stdout) if r.stdout.strip() else {}

def tk_year(*args: str) -> dict:
    return tk(*args, "--year", str(ANNO))

def tk_schema(layer: str) -> dict:
    cmd = ["toolkit", "inspect", "schema", "-l", layer,
           str(CFG_PATH), "--json", "--year", str(ANNO)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print(r.stderr, file=sys.stderr)
        return {"columns": 0, "schema": []}
    return json.loads(r.stdout) if r.stdout.strip() else {}

def show(df) -> None:
    try:
        display(df)
    except NameError:
        print(df.to_string())

WORKSPACE_ROOT = CANDIDATE_DIR.parent.parent
def _rel(p: Path) -> str:
    try: return str(p.relative_to(WORKSPACE_ROOT))
    except ValueError: return str(p)

print(f"slug: {SLUG}    anno: {ANNO}")
print(f"config: {_rel(CFG_PATH)}")

paths = tk_year("inspect", "paths")
P = paths.get("paths", {})
RAW_DIR = Path(P.get("raw", {}).get("dir", ""))
CLEAN_DIR = Path(P.get("clean", {}).get("dir", ""))
MART_DIR = Path(P.get("mart", {}).get("dir", ""))
print(f"raw:   {_rel(RAW_DIR)}")
print(f"clean: {_rel(CLEAN_DIR)}")
print(f"mart:  {_rel(MART_DIR)}")

## SQL di riferimento

Le SQL sono la fonte di verita per capire cosa deve contenere ogni layer.

In [ ]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()

## Quality gates

Riassunto del run e validazione.

In [ ]:
run_info = paths.get("latest_run", {})
print(f"Ultimo run: {run_info.get('status','?')}  (ID: {run_info.get('run_id','?')})")
print()

schema_clean = tk_schema("clean")
cols_clean = schema_clean.get("schema", [])
print(f"Clean: {schema_clean.get('columns', 0)} colonne")
for c in cols_clean[:7]:
    print(f"  {c.get('name','?'):30s} {c.get('type','?')}")
if len(cols_clean) > 7: print(f"  ... ({len(cols_clean)} totali)")
print()

profiles = paths.get("layer_profiles", {})
clean_prof = profiles.get("clean_output", {})
if clean_prof:
    print(f"Clean: {clean_prof.get('row_count','?')} righe, {clean_prof.get('columns_count','?')} colonne")
for tbl in profiles.get("mart_tables", []):
    print(f"Mart: {tbl.get('row_count','?')} righe ({tbl.get('name','?')})")
for trans in profiles.get("clean_to_mart", []):
    print(f"Transizione: {trans.get('source_row_count','?')} -> {trans.get('target_row_count','?')} righe")
print()

rh = paths.get("raw_hints", {})
print(f"Raw: {rh.get('primary_output_file','?')}  |  encoding={rh.get('encoding','?')}  delim={rh.get('delim','?')}")

## Analisi dati

Caricamento del mart e analisi esplorativa.


In [ ]:
con = duckdb.connect()

mart_files = list(MART_DIR.glob("*.parquet")) if MART_DIR.exists() else []
for mf in sorted(mart_files):
    table_name = mf.stem
    p = str(mf).replace("'", "''")
    con.execute(f"CREATE OR REPLACE VIEW {table_name} AS SELECT * FROM read_parquet('{p}')")
    cnt = con.execute(f"SELECT count(*) FROM {table_name}").fetchone()[0]
    print(f"{table_name}: {cnt} righe")

if not mart_files:
    print("Nessun mart disponibile. Esegui: toolkit run mart")

In [ ]:
print("=== mart_sbarchi_mensili: ultimi 12 mesi ===")
df = con.execute("""
    SELECT * FROM mart_sbarchi_mensili
    ORDER BY mese DESC LIMIT 12
""").fetchdf()
show(df)

In [ ]:
for tbl_name in ['mart_sbarchi_mensili']:
    if tbl_name not in con.execute("SELECT table_name FROM information_schema.tables").fetchdf()['table_name'].values:
        continue
    cols = con.execute(f"DESCRIBE {tbl_name}").fetchall()
    exprs = []
    for c in cols[:8]:
        safe = c[0].replace('"', '""')
        exprs.append(f'round(100.0 * sum(CASE WHEN "{safe}" IS NULL THEN 1 ELSE 0 END) / count(*), 2) as "null_{c[0]}"')
    if exprs:
        q = f"SELECT count(*) as tot, {', '.join(exprs)} FROM {tbl_name}"
        nulls = con.execute(q).fetchdf()
        print(f"\n--- {tbl_name} ---")
        show(nulls)


## Boundary e note

Perimetro, esclusioni, anomalie note.

In [ ]:
PERIMETRO = "Arrivi via mare 2019-2025 (onData)"

print(f"Slug      : {SLUG}")
print(f"Anno run  : {ANNO}")
print(f"Perimetro : {PERIMETRO}")
print()
print("Note boundary:")
print("  - onData si e' fermato a settembre 2025 (gap 2025-2026)")
print("  - dati estratti da PDF del Ministero dell'Interno")


## Verdetto

Riepilogo end-to-end della pipeline.

In [ ]:
run_info = paths.get("latest_run", {})
run_status = run_info.get("status", "?")
profiles = paths.get("layer_profiles", {})
clean_p = profiles.get("clean_output", {})
mart_tables = profiles.get("mart_tables", [])
transitions = profiles.get("clean_to_mart", [])

status_icon = {'SUCCESS': 'OK', 'FAILED': 'FAIL', 'IN_PROGRESS': '...'}.get(run_status, '?')
print(f"Pipeline  : {status_icon} {run_status}")
print()
if clean_p:
    print(f"  Clean: {clean_p.get('row_count', '?'):>8} righe, {clean_p.get('columns_count', '?')} colonne")
for tbl in mart_tables:
    print(f"  Mart:  {tbl.get('row_count', '?'):>8} righe ({tbl.get('name', '?')})")
for t in transitions:
    print(f"  Transizione: {t.get('source_row_count','?')} -> {t.get('target_row_count','?')} righe")
print()
print("Notebook v0 completato.")